# CARA-T-inspired retraining experiment ? CoverType

This notebook presents the binary CoverType experiment in the Airline reference notebook?s research flow: data validation, baseline model, paper-to-project mapping, validation-only calibration and cost sensitivity, sequential holdout replay, audit, and conclusions.

**Saved-artifact analysis is the default.** The final cell offers an explicit full-pipeline rerun. This work reuses the maintained CoverType scripts and does not create a second policy implementation.

## 1. Scope and research question

The experiment studies whether to keep or retrain a binary forest-cover classifier as ordered batches are evaluated. The supplied input is `covtype.libsvm.binary.scale.bz2`: 581,012 rows, 54 already-scaled features, and original labels 1 and 2 mapped to 0 and 1. This is not the standard seven-class CovType task.

The batches preserve file order only; the source has no timestamp, so they are not verified chronological periods and do not prove drift.

## 2. Imports, paths, and execution mode

This notebook loads the prepared sparse data and saved results by default. The project?s canonical pipeline remains in `src/01_...py` through `src/08_...py`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "src" / "covtype_pipeline.py").is_file():
            return base.resolve()
        candidate = base / "Covertype"
        if (candidate / "src" / "covtype_pipeline.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Covertype project directory from the notebook working directory.")


PROJECT_DIR = find_project_root()
WORKSPACE_ROOT = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))
from covtype_pipeline import load_processed_data

DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
CARA_DIR = ARTIFACTS_DIR / "cara_t"
HOLDOUT_DIR = CARA_DIR / "test_holdout"
print("Project:", PROJECT_DIR)
print("Workspace input:", WORKSPACE_ROOT / "covtype.libsvm.binary.scale.bz2")

## 3. Validate the prepared sparse dataset and split

Preparation partitions 100 contiguous batches without crossing split boundaries: batches 0?69 train, 70?84 validation, and 85?99 holdout replay. The project loader checks feature/target shapes, labels, manifest continuity, and split counts.

In [ ]:
features, labels, manifest, stream_metadata = load_processed_data()
if features.shape != (581_012, 54) or len(labels) != 581_012:
    raise ValueError(f"Unexpected prepared data shape: X={features.shape}, y={labels.shape}")
if manifest["batch_id"].tolist() != list(range(100)):
    raise ValueError("Expected batch IDs 0 through 99 exactly once and in order.")
if manifest["split"].value_counts().to_dict() != {"train": 70, "validation": 15, "test": 15}:
    raise ValueError("Expected a 70/15/15 train/validation/holdout split.")

print(f"Sparse feature matrix: {features.shape}; stored nonzero entries: {features.nnz:,}")
print("Feature count:", stream_metadata["feature_count"])
print("Target mapping:", stream_metadata["target"])
display(manifest.groupby("split", sort=False).agg(
    batches=("batch_id", "count"),
    records=("n_records", "sum"),
    class_0=("class_0", "sum"),
    class_1=("class_1", "sum"),
))

## 4. Descriptive inspection

The target-rate chart reflects this file?s batch index. Without timestamps or verified ordering, it must not be interpreted as measured temporal drift.

In [ ]:
batch_summary = manifest.copy()
batch_summary["positive_rate"] = batch_summary["class_1"] / batch_summary["n_records"]
display(batch_summary.groupby("split", sort=False).agg(
    batches=("batch_id", "count"),
    records=("n_records", "sum"),
    mean_positive_rate=("positive_rate", "mean"),
    min_batch_size=("n_records", "min"),
    max_batch_size=("n_records", "max"),
).round(4))

ax = batch_summary.plot(x="batch_id", y="positive_rate", figsize=(11, 3.5), legend=False, title="CoverType binary positive-class rate by file-order batch")
ax.set_ylabel("Positive-class rate")
ax.set_xlabel("Batch index (file order; not verified time)")
plt.tight_layout()
plt.show()

## 5. Model V1 and baseline metrics

Model V1 is a Keras binary classifier with 54 inputs, Dense(32, ReLU), Dropout(0.2), Dense(16, ReLU), and sigmoid output. It is trained on batches 0?69 with early stopping against batches 70?84. The saved metrics below are validation metrics from the model metadata, not holdout policy metrics.

In [ ]:
metadata_path = ARTIFACTS_DIR / "model_v1" / "metadata.json"
with metadata_path.open(encoding="utf-8") as file:
    model_metadata = json.load(file)
print("Algorithm:", model_metadata["algorithm"])
print("Training batches:", model_metadata["training_batches"])
print("Validation batches:", model_metadata["validation_batches"])
print("Training / validation rows:", model_metadata["training_records"], "/", model_metadata["validation_records"])
display(pd.Series(model_metadata["metrics"], name="Model V1 validation").to_frame().round(4))

## Paper method and this project's adaptation

The supplied paper formalizes the trade-off between model staleness and retraining cost (Mahadevan & Mathioudakis, 2024, Sections 4?5; the threshold variant is CARA-T, Section 5.1). Its experimental protocol uses an offline phase to select a threshold and then an online retraining strategy. This project borrows that structure, but its objective and implementation are project-specific, not a complete reproduction of the paper.

For intuition, an RBF similarity is

\[
K(q,x)=\exp(-\gamma\lVert q-x\rVert^2),
\]

and a similarity-weighted 0?1 loss proxy is

\[
C_t(M)=\operatorname{mean}_{q\in Q_t,\,(x,y)\in D_t}
       K(q,x)\,\mathbf{1}[M(x)\ne y].
\]

The implementation compares a current-batch proxy with the active model's reference proxy:

\[
\Delta_t=C_t(M_t)-C_{\mathrm{ref}}(M_t),\qquad
\Delta_t\ge\tau\;\Longrightarrow\;\text{RETRAIN},
\quad\Delta_t<\tau\;\Longrightarrow\;\text{KEEP}.
\]

Its accounting assigns the assumed \(\kappa\) to RETRAIN and the batch proxy cost to KEEP. Neither quantity is measured business loss, money, or compute. Because the loss uses true labels, decisions in a real system would wait for labels.

**Paper source:** [cost1-aware retrining algo.pdf](../../cost1-aware%20retrining%20algo.pdf), Mahadevan & Mathioudakis, "Cost-aware retraining for machine learning," *Knowledge-Based Systems* 293 (2024), 111610. Consult Sections 4-5 for cost definitions/CARA-T and Section 7 for experimental setup. The Airline notebook is used as the project-format reference; this notebook reuses the Electricity implementation rather than copying the Airline model/data pipeline.

## 6. Validation-only calibration and \(\kappa\) sensitivity

Candidate thresholds are derived from standalone Model V1 validation staleness, including \(-\infty\) and \(+\infty\) endpoints. For each candidate \(\kappa\) multiplier, the optimizer sequentially replays validation batches; retrained models and their reference costs affect subsequent validation decisions. The holdout batches are not used to select the threshold.

In [ ]:
threshold_path = CARA_DIR / "best_thresholds.csv"
threshold_search_path = CARA_DIR / "threshold_search.csv"
kappa_path = CARA_DIR / "cost_sensitivity" / "best_policy_per_kappa.csv"
for path in (threshold_path, threshold_search_path, kappa_path):
    if not path.is_file():
        raise FileNotFoundError(f"Calibration artifact is missing: {path}")

best_thresholds = pd.read_csv(threshold_path)
threshold_search = pd.read_csv(threshold_search_path)
kappa_sweep = pd.read_csv(kappa_path)
selected = best_thresholds.loc[np.isclose(best_thresholds["kappa_multiplier"], 1.0)]
if len(selected) != 1:
    raise ValueError("Expected exactly one selected threshold at kappa multiplier 1.0.")
print("Default validation selection:")
display(selected)
print("Best validation policy by assumed kappa:")
display(kappa_sweep)

ax = kappa_sweep.plot(x="kappa", y="retrain_count", marker="o", figsize=(7, 3.5), legend=False, title="Selected validation retrains vs assumed kappa")
ax.set_ylabel("Validation retrains")
plt.tight_layout()
plt.show()

## 7. Sequential holdout protocol

For each holdout batch, the active model predicts and is scored first. If selected, a replacement model is trained from fresh weights on the current batch and used for later batches. The final holdout batch's retraining decision is logged after scoring and cannot affect any holdout prediction. The policies are Never, Always, Periodic (every 3 batches), and CARA-T-inspired.

## 8. Recorded holdout comparison

The table below is loaded from the saved project summary. Accuracy/F1 assess predictions; proxy cost and retraining count assess the defined decision objective. The preferred policy depends on which objective matters.

In [ ]:
summary_path = HOLDOUT_DIR / "holdout_strategy_summary.csv"
batch_results_path = HOLDOUT_DIR / "holdout_batch_results.csv"
if not summary_path.is_file() or not batch_results_path.is_file():
    raise FileNotFoundError("Saved CoverType holdout summary or batch results are missing.")

summary = pd.read_csv(summary_path)
batch_results = pd.read_csv(batch_results_path)
expected_strategies = {"Never retrain", "Always retrain", "Periodic (every 3 batches)", "CARA-T-inspired"}
if set(summary["strategy"]) != expected_strategies:
    raise ValueError(f"Unexpected strategy set: {summary['strategy'].tolist()}")
if not batch_results.groupby("strategy")["batch_id"].nunique().eq(15).all():
    raise ValueError("Every strategy must cover holdout batches 85?99.")

display(summary[["strategy", "retrain_count", "accuracy", "f1", "total_proxy_cost"]].sort_values("strategy").round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.sort_values("total_proxy_cost").plot.bar(x="strategy", y="total_proxy_cost", ax=axes[0], legend=False, title="Total proxy cost (lower is better)")
summary.sort_values("accuracy", ascending=False).plot.bar(x="strategy", y="accuracy", ax=axes[1], legend=False, title="Holdout accuracy")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 9. Batch-level replay and audit

The saved audit verifies decision rules, model-version progression, batch coverage, and proxy-cost reconciliation. These checks support implementation consistency, not real-world policy optimality.

In [ ]:
audit_path = HOLDOUT_DIR / "holdout_audit_summary.csv"
if not audit_path.is_file():
    raise FileNotFoundError(f"Holdout audit summary is missing: {audit_path}")
audit = pd.read_csv(audit_path)
boolean_checks = ["batch_coverage_ok", "decision_rule_ok", "model_version_progression_ok", "proxy_cost_reconciles", "summary_reconciles"]
missing_checks = set(boolean_checks) - set(audit.columns)
if missing_checks:
    raise ValueError(f"Audit summary lacks expected check columns: {sorted(missing_checks)}")
assert audit[boolean_checks].astype(bool).all().all()
assert audit["distinct_batch_count"].eq(15).all()
print("PASS: every saved policy audit check is true and covers all 15 holdout batches.")
display(audit)

fig, ax = plt.subplots(figsize=(11, 4))
for strategy, rows in batch_results.groupby("strategy", sort=False):
    ax.plot(rows["batch_id"], rows["accuracy"], marker=".", label=strategy)
ax.set(title="CoverType holdout accuracy by file-order batch", xlabel="Batch ID (not verified time)", ylabel="Accuracy")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

## 10. Findings and limitations

In this saved run, Always retrain achieves the strongest holdout predictive metrics. CARA-T-inspired retraining records the lowest defined proxy cost, with 11 retrains versus 15 for Always. The conclusion is conditional on this split, one run/seed, the binary dataset variant, and the proxy-cost definition.

The source is not the original seven-class task and has no timestamps. The assumed \(\kappa\) is not measured cost; error-based staleness needs labels; retraining uses only the current batch and may forget older patterns. The notebook is an offline replay, not a production service.

## 11. Reproduction

The default notebook analyzes the saved artifacts. To regenerate them, the workspace-root compressed LIBSVM file must exist. The opt-in cell runs the project's canonical scripts in dependency order; this is compute-intensive and overwrites generated outputs. Threshold calibration uses validation data; do not tune on holdout.

In [ ]:
# Explicit opt-in; reruns preparation, training, calibration, holdout replay, and audit.
RUN_FULL_PIPELINE = False
PIPELINE_STEPS = [
    "01_inspect_data.py",
    "02_prepare_batches.py",
    "03_train_baseline.py",
    "08_evaluate_batches.py",
    "04_compute_staleness.py",
    "05_optimize_policy.py",
    "06_evaluate_holdout.py",
    "07_audit_results.py",
]

if RUN_FULL_PIPELINE:
    if not (WORKSPACE_ROOT / "covtype.libsvm.binary.scale.bz2").is_file():
        raise FileNotFoundError("Restore covtype.libsvm.binary.scale.bz2 at the workspace root before a full rerun.")
    for script in PIPELINE_STEPS:
        print(f"Running {script} ...")
        subprocess.run([sys.executable, str(PROJECT_DIR / "src" / script)], cwd=WORKSPACE_ROOT, check=True)
    print("Full CoverType pipeline and audit completed. Re-run result-loading cells above to refresh their displays.")
else:
    print("No training was started. Set RUN_FULL_PIPELINE=True to rerun the canonical pipeline.")